In [19]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import scipy.stats as stats
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import zero_one_loss
from tqdm import tqdm

In [2]:
rng = np.random.default_rng(seed=4)

# Questão 2

## item a)

Dadas as distribuições:

$$
P(Y = A) = 0{,}5,\qquad X \mid Y=A \sim U(0,4)
$$

$$
P(Y = B) = 0{,}5,\qquad X \mid Y=B \sim N(2,1)
$$

A densidade condicional de \(X\) dado \(Y=A\) é

$$
p(x \mid Y=A)=
\begin{cases}
\dfrac{1}{4}, & x\in(0,4],\\[6pt]
0, & \text{caso contrário.}
\end{cases}
$$

A densidade condicional de \(X\) dado \(Y=B\) é

$$
p(x \mid Y=B)=\dfrac{1}{\sqrt{2\pi}}\exp\!\left(-\dfrac{(x-2)^2}{2}\right).
$$

A probabilidade a posteriori é dada por

$$
P(Y=A\mid x)=\frac{p(x\mid Y=A)\,P(Y=A)}{p(x)},
$$

onde

$$
p(x)=p(x\mid Y=A)P(Y=A)+p(x\mid Y=B)P(Y=B).
$$

Para $x\in[0,4]$ temos $p(x\mid Y=A)=\tfrac14$, logo

$$
P(Y=A\mid x)
=\frac{\tfrac14\cdot\tfrac12}{\tfrac14\cdot\tfrac12+\;p(x\mid Y=B)\cdot\tfrac12}
= \frac{\tfrac14}{\tfrac14+p(x\mid Y=B)}.
$$

Queremos agora a probabilidade posterior:

$$
P(Y=B \mid x)
= \frac{p(x \mid Y=B)\,P(B)}
       {p(x \mid Y=B)P(B) + p(x \mid Y=A)P(A)}
$$

Para $x\in[0,4]$ temos $p(x\mid Y=A)=\tfrac14$, logo

$$
P(Y=B \mid x)
= \frac{0.5\,p(x\mid B)}{0.25\cdot 0.5 + 0.5\,p(x\mid B)}
= \frac{p(x\mid B)}{0.25 + p(x\mid B)}
$$

Assim,

$$
\frac{P(Y=B \mid x)}{P(Y=A \mid x)}
= 4\,p(x\mid B)
$$

Usando a densidade normal:

$$
4\,p(x\mid B)
= \frac{4}{\sqrt{2\pi}} \exp\!\left(-\frac{(x-2)^2}{2}\right)
$$

A regra de decisão Bayesiana diz:

$$
\hat{Y}=B \quad \Longleftrightarrow \quad P(Y=B\mid x)>P(Y=A\mid x)
$$

Ou seja:

$$
4\,p(x\mid B) > 1
$$

Substituindo:

$$
\frac{4}{\sqrt{2\pi}}\, e^{-\frac12 (x-2)^2} > 1
$$

$$
e^{-\frac12 (x-2)^2} > \frac{\sqrt{2\pi}}{4}
$$

Tomando logaritmo:

$$
-\frac12 (x-2)^2 > \ln\sqrt{2\pi} - \ln 4
$$

$$
(x-2)^2 < 2(\ln 4 - \tfrac12\ln 2\pi)
$$

Calculando:

$$
|x - 2| < 0.967
$$

Portanto, a região classificada como classe B é:

$$
1.033 < x < 2.967
$$

A probabilidade de erro é a probabilidade de x estar fora do intervalo calculado e ser da classe B, mais a probabilidaded de x estar no intervalo e ser da classe A:

\begin{aligned}
P(\text{erro}) &= 
P\big(x < 1.033 \mid Y=B\big)P(B)
+ P\big(x > 2.967 \mid Y=B\big)P(B)
\\[6pt]
&\quad + P\big(1.033 < x < 2.967 \mid Y=A\big)P(A)
\end{aligned}

Como $X\mid A \sim U(0,4)$:

$$
P(1.033 < x < 2.967 \mid A)
= \int_{1.033}^{2.967} \frac14 \, dx
= \frac{1}{4}(2.967 - 1.033)
= 0.4835
$$

Assim:

$$
P(\text{erro})
= 0.5\cdot 0.4835 + 0.5\left( P(X>2.967) + P(X<1.033) \right)
$$

Usando a normal padrão:

$$
P(X>2.967 \mid B) = 1 - \Phi(0.967) = 0.167
$$

$$
P(X<1.033 \mid B) = \Phi(-0.967) = 0.167
$$

Logo:

$$
P(\text{erro})
= 0.5\cdot 0.4835 + 0.5\cdot(0.167 + 0.167)
$$

$$
P(\text{erro}) = 0.386
$$

Portanto:

$$
\boxed{P(\text{erro}) \approx 38.6\%}
$$



## item b)
O erro esperado de uma árvore de decisão com altura 1 e duas folhas que usa o critério de Gini para conjuntos de treino de tamanho 50 é 47,6%. Já para conjuntos de treino de tamanho 100 é 47,2%. O erro no conjunto de treino maior é mais baixo, embora ambos sejam maior que do classificador do item anterior.

Para calcular o erro esperado, foram feitas 1000 simulações com amostras de tamanho 50 e 100. Em cada simulação, foi treinada uma árvore de decisão com altura 1 e duas folhas, e calculado o erro verdadeiro usando um conjunto de teste de tamanho 200.000.

In [14]:
n_sim = 1_000

samples_50 = np.zeros((n_sim, 50))
y_50 = np.empty((n_sim, 50), dtype='<U1')

samples_100 = np.zeros((n_sim, 100))
y_100 = np.empty((n_sim, 100), dtype='<U1')

In [15]:
for i in range(n_sim):
    #sorteio da classe
    y_50[i,:] = rng.choice(['A', 'B'], size=50, p=[0.5, 0.5])
    # sorteio dos dados
    maskA = (y_50[i,:] == 'A')
    samples_50[i,maskA] = rng.uniform(0.0, 4.0, size=maskA.sum())
    maskB = ~maskA
    samples_50[i,maskB] = rng.normal(loc=2.0, scale=1.0, size=maskB.sum())
    
    #sorteio da classe
    y_100[i,:] = rng.choice(['A', 'B'], size=100, p=[0.5, 0.5])
    # sorteio dos dados
    maskA = (y_100[i,:] == 'A')
    samples_100[i,maskA] = rng.uniform(0.0, 4.0, size=maskA.sum())
    maskB = ~maskA
    samples_100[i,maskB] = rng.normal(loc=2.0, scale=1.0, size=maskB.sum())

In [20]:
def sample_test(n_test=200_000):
    y = rng.choice(['A','B'], size=n_test, p=[0.5, 0.5])
    x = np.empty(n_test)
    maskA = (y == 'A')
    x[maskA] = rng.uniform(0.0, 4.0, size=maskA.sum())
    maskB = ~maskA
    x[maskB] = rng.normal(2.0, 1.0, size=maskB.sum())
    return x.reshape(-1,1), y

In [33]:
X_test, y_test = sample_test(n_test=200_000)

true_errors_50 = np.zeros(n_sim)
true_errors_100 = np.zeros(n_sim)

for i in tqdm(range(n_sim)):
    X_train = samples_50[i].reshape(-1,1)
    y_train = y_50[i]

    clf = DecisionTreeClassifier(criterion='gini', max_depth=1)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    true_errors_50[i] = zero_one_loss(y_test, y_pred)

    X_train = samples_100[i].reshape(-1,1)
    y_train = y_100[i]

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    true_errors_100[i] = zero_one_loss(y_test, y_pred)


print(f"True error esperado (amostras de tamanho 50): {true_errors_50.mean():.3}")
print(f"True error esperado (amostras de tamanho 100): {true_errors_100.mean():.3f}")

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:57<00:00,  8.51it/s]

True error esperado (amostras de tamanho 50): 0.476
True error esperado (amostras de tamanho 100): 0.472


## item c)
O erro esperado de uma árvore de decisão com altura 2 e 3 folhas que usa o critério de Gini para conjuntos de treino de tamanho 50 é 45,9%. Já para conjuntos de treino de tamanho 100 é 45,0%. A árvore com altura 2 possui melhor desempenho que a árvore com altura 1, embora ainda seja pior que o classificador ótimo de bayes.

In [34]:
X_test, y_test = sample_test(n_test=200_000)

true_errors_50 = np.zeros(n_sim)
true_errors_100 = np.zeros(n_sim)

for i in tqdm(range(n_sim)):
    X_train = samples_50[i].reshape(-1,1)
    y_train = y_50[i]

    clf = DecisionTreeClassifier(criterion='gini', max_depth=2, max_leaf_nodes=3)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    true_errors_50[i] = zero_one_loss(y_test, y_pred)

    X_train = samples_100[i].reshape(-1,1)
    y_train = y_100[i]

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    true_errors_100[i] = zero_one_loss(y_test, y_pred)

print(f"True error esperado (amostras de tamanho 50): {true_errors_50.mean():.3f}")
print(f"True error esperado (amostras de tamanho 100): {true_errors_100.mean():.3f}")

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [02:04<00:00,  8.02it/s]

True error esperado (amostras de tamanho 50): 0.459
True error esperado (amostras de tamanho 100): 0.450


## item d)
O erro esperado de uma árvore de decisão com altura 2 e 4 folhas que usa o critério de Gini para conjuntos de treino de tamanho 50 é 45,8%. Já para conjuntos de treino de tamanho 100 é 44,6%. A árvore com altura 2 e 4 folhas possui desempenho melhor que a árvore com altura 2 e 3 folhas em conjuntos de teste maiores. Em conjuntos menores, não há diferença significativa entre as duas árvores. Ambas ainda são piores que o classificador ótimo de bayes.

In [35]:
X_test, y_test = sample_test(n_test=200_000)

true_errors_50 = np.zeros(n_sim)
true_errors_100 = np.zeros(n_sim)

for i in tqdm(range(n_sim)):
    X_train = samples_50[i].reshape(-1,1)
    y_train = y_50[i]

    clf = DecisionTreeClassifier(criterion='gini', max_depth=2, max_leaf_nodes=4)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    true_errors_50[i] = zero_one_loss(y_test, y_pred)

    X_train = samples_100[i].reshape(-1,1)
    y_train = y_100[i]

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    true_errors_100[i] = zero_one_loss(y_test, y_pred)

print(f"True error esperado (amostras de tamanho 50): {true_errors_50.mean():.3f}")
print(f"True error esperado (amostras de tamanho 100): {true_errors_100.mean():.3f}")

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [02:48<00:00,  5.93it/s]

True error esperado (amostras de tamanho 50): 0.458
True error esperado (amostras de tamanho 100): 0.446


## item e)
O erro esperado da melhor árvore de decisão possível é 44,6%, pois minimiza 

# Questão 5
## item a)
Dado o kernel Gaussiano:

$$
K(x_i, x_j) = e^{-\frac{1}{2}\lVert x_i - x_j\rVert^2}
$$

Queremos calcular:

$$
\lVert \phi(x_i) - \phi(x_j) \rVert^2
$$

Sabemos que:

$$
\lVert \phi(x_i) - \phi(x_j) \rVert^2
= (\phi(x_i) - \phi(x_j))^\top (\phi(x_i) - \phi(x_j))
$$

Expandindo:

$$
\lVert \phi(x_i) - \phi(x_j) \rVert^2
= \phi(x_i)^\top \phi(x_i)
+ \phi(x_j)^\top \phi(x_j)
- 2\,\phi(x_i)^\top \phi(x_j)
$$

Usando o truque do kernel:

$$
\phi(x_i)^\top \phi(x_i) = K(x_i, x_i) = e^{-\frac12\lVert x_i - x_i\rVert^2} = e^0 = 1
$$

$$
\phi(x_j)^\top \phi(x_j) = K(x_j, x_j) = 1
$$

E:

$$
\phi(x_i)^\top \phi(x_j) = K(x_i, x_j) = e^{-\frac12\lVert x_i - x_j\rVert^2}
$$

Portanto:

$$
\lVert \phi(x_i) - \phi(x_j) \rVert^2
= 1 + 1 - 2K(x_i, x_j)
$$

Ou seja:

$$
\lVert \phi(x_i) - \phi(x_j) \rVert^2
= 2 - 2 K(x_i, x_j)
$$

Como sempre temos $K(x_i, x_j) > 0$ para o kernel Gaussiano:

$$
\lVert \phi(x_i) - \phi(x_j) \rVert^2 < 2
$$


## item b)
Para z um ponto qualquer, a predição do SVM é dada por:

$$
\hat{y}_z = \operatorname{sign}(w^\top \phi(z) + w_0)
$$

Usando a expansão em termos dos vetores de suporte:

$$
\hat{y}_z = 
\operatorname{sign}\!\left( 
\sum_{i \in SV} y_i \alpha_i\, K(x_i, z) + w_0
\right)
$$

O kernel Gaussiano é:

$$
K(x_i, z) = e^{-\frac12 \lVert x_i - z\rVert^2}
$$

Quando z está muito distante de todos os vetores de suporte $x_i$, temos:

$$
K(x_i, z) \to 0 \qquad \forall\, i \in SV
$$

Portanto:

$$
\hat{y}_z = \operatorname{sign}(w_0)
$$

Logo, para dois pontos z e z' muito distantes de qualquer ponto do conjunto de treino e consequentemente dos vetores de suporte, a classificação será a mesma, dada por $\operatorname{sign}(w_0)$

## item c)
A função de restrição para um ponto $x_j$ é:

$$
y_j (w^\top \phi(x_j) + b)
$$

Usando a expansão do vetor \(w\):

$$
w = \sum_{i \in SV} y_i \alpha_i\, \phi(x_i)
$$

Substituindo:

$$
y_j (w^\top \phi(x_j) + b)
= y_j \left( 
\sum_{i \in SV} y_i \alpha_i\, K(x_i, x_j) + b
\right)
$$

Separando o termo \(i=j\):

$$
= \alpha_j \;+\;
\sum_{\substack{i \in SV \\ i \ne j}}
y_i y_j \alpha_i\, K(x_i, x_j)
\;+\; y_j b
$$

Quando $C \to \infty$, o kernel Gaussiano satisfaz:

$$
K(x_i, x_j) \to 0 \qquad \text{para } i \ne j
$$

Portanto:

$$
y_j (w^\top \phi(x_j) + b)
= \alpha_j + b\,y_j
$$

Logo, basta escolher:

$$
\alpha_j > b\, y_j \qquad \forall j
$$

Assim, na versão hard do SVM com kernel RBF estreito, é possível escolher os parâmetros de forma que a restrição seja satisfeita para todos os pontos do conjunto de treino, ou seja, que o erro de treino seja zero. Como essa classe de hipóteses consegue se adequar a qualquer conjunto de treino com pontos distintos de dimensão m, ela shatter esses conjuntos para qualquer valor de m. Logo, a dimensão VC do SVM com kernel gaussiano é infinita.
